# SVM Classification - Intensity-Only Mode Test

**Testing intensity-only classification:**
- Uses **mean intensity** across wavelength range as the ONLY feature
- No spectral information - purely based on brightness
- Compare performance against spectral classification

**Complete pipeline with proper naming:**
1. **Training** → `training_*` ROIs train the SVM (using intensity only)
2. **Classification** → SVM outputs `classified_*` predictions (using intensity only)
3. **Filtering** → Small clusters become `filtered_*`
4. **Merge** → `filtered_* + classified_sediment` → `new_sediment`
5. **Validation** → Compare 3 classes against `validation_*` ROIs

**Key features:**
- ✅ **Intensity-only mode**: `use_intensity_only=True`
- ✅ Clear naming throughout the pipeline
- ✅ No empty strings - all pixels classified
- ✅ Can visualize 5-class filtered map
- ✅ Morphological operations disabled

## Setup and Import

In [ ]:
import importlib
import sys
import os
import numpy as np

sys.path.append(os.path.abspath("../"))

from utils.gref_pipeline import georef
from gref_pipeline import config

importlib.reload(georef)
from utils.gref_pipeline.georef import *

print("✅ Module loaded successfully!")

## Load Data and ROIs

In [ ]:
# Load transect
transect = load_transect(config.OUTPUT_FOLDER)
transect.list_files()

# Select file
cube = transect.select_files(["rad_uhi_20241029_115057_5"])
cube.describe()

In [ ]:
# Apply illumination correction
cube.apply_illumination_correction_v2()

In [ ]:
# Import ROIs
cube.import_rois("./ROIs/057_5_combined.json")
cube.list_rois()

In [ ]:
# Define ROI collections
training_rois = [
    "training_dark",
    "training_sediment",
    "training_bombs",
]

validation_rois = [
    "validation_dark",
    "validation_sediment",
    "validation_bombs",
]

print(f"Training ROIs: {training_rois}")
print(f"Validation ROIs: {validation_rois}")

In [ ]:
# Test: No normalization (raw data with offset visible)
cube.plot_spectrum(
    roi_names=training_rois,
    use_corrected=True,
    wavelength_range=(490, 680),
    wavelength_smoothing=10,
    # interpolate_wavelengths=[35, 104, 157],
    interpolate_wavelengths=[64, 107, 138],
    normalize_method=None,  # No normalization
    legend_loc="outside",
    use_inline_labels=False,
    show_std=True,
)

In [ ]:
# Plot validation ROIs
%matplotlib inline

cube.plot_georef(
    apply_alignment_shift=True,
    use_corrected=True,
    coordinate_system="NED",
    track_start=config.UHI_TRACK_RANGE_5[0],
    track_end=config.UHI_TRACK_RANGE_5[1],
    figsize=(30, 10),
    roi_collection=validation_rois,
    roi_marker_size=2,
    roi_legend_loc="outside",
    roi_marker_edgewidth=0,
    roi_legend_markersize=50,
)

## Train SVM with Cross-Validation

In [ ]:
# Train SVM with spatial cross-validation - INTENSITY ONLY MODE
cv_results = cube.train_svm_with_cv(
    training_rois=training_rois,
    segment_start=config.UHI_TRACK_RANGE_5[0],
    segment_end=config.UHI_TRACK_RANGE_5[1],
    wavelength_range=(490, 680),  # Still used to filter which wavelengths to average
    cv_folds=5,
    use_corrected=True,
    svm_kernel="rbf",
    optimize_params=True,
    add_brightness_feature=False,
    use_intensity_only=True,  # 🔥 NEW: Use only mean intensity!
    quiet=False,
)

# Print summary
print(f"\n" + "=" * 60)
print(f"📊 CROSS-VALIDATION SUMMARY (INTENSITY-ONLY)")
print(f"=" * 60)
print(
    f"Accuracy:  {cv_results['cv_mean_metrics']['accuracy_mean']:.3f} ± {cv_results['cv_mean_metrics']['accuracy_std']:.3f}"
)
print(
    f"Precision: {cv_results['cv_mean_metrics']['precision_mean']:.3f} ± {cv_results['cv_mean_metrics']['precision_std']:.3f}"
)
print(
    f"Recall:    {cv_results['cv_mean_metrics']['recall_mean']:.3f} ± {cv_results['cv_mean_metrics']['recall_std']:.3f}"
)
print(
    f"F1 Score:  {cv_results['cv_mean_metrics']['f1_mean']:.3f} ± {cv_results['cv_mean_metrics']['f1_std']:.3f}"
)

print(f"\n🎯 Best hyperparameters:")
print(f"  C = {cv_results['best_params']['C']}")
print(f"  gamma = {cv_results['best_params']['gamma']}")

print(f"\n📍 Training pixels used:")
for class_name, count in cv_results["training_pixels_per_class"].items():
    print(f"  {class_name}: {count} pixels")

In [ ]:
# Visualize filtered training ROIs
cube.plot_georef(
    use_corrected=True,
    figsize=(40, 10),
    roi_collection=cv_results["filtered_training_rois"],
    roi_marker_size=3,
    roi_legend_loc="outside",
    roi_marker_edgewidth=0,
    roi_legend_markersize=40,
)

In [ ]:
# Plot spatial groups created during training
if (
    "spatial_groups_roi_collection" in cv_results
    and cv_results["spatial_groups_roi_collection"]
):
    cube.plot_georef(
        use_corrected=True,
        figsize=(40, 10),
        roi_collection=cv_results["spatial_groups_roi_collection"],
        roi_marker_size=3,
        roi_legend_loc="outside",
        roi_marker_edgewidth=0,
        roi_legend_markersize=40,
    )

    # Print group info
    print("\n📊 Spatial Groups Created:")
    for group_name, pixels in sorted(
        cv_results["spatial_groups_roi_collection"].items()
    ):
        print(f"   {group_name}: {len(pixels)} pixels")
else:
    print(
        "⚠️ No spatial groups found in cv_results. Training may not have used spatial clustering."
    )

In [ ]:
# Plot spectra for all spatial groups
if (
    "spatial_groups_roi_collection" in cv_results
    and cv_results["spatial_groups_roi_collection"]
):
    # Temporarily add spatial groups to ROI collection
    original_rois = cube.roi_collection.copy()
    cube.roi_collection.update(cv_results["spatial_groups_roi_collection"])

    # Plot spectra
    cube.plot_spectrum(
        roi_names=list(cv_results["spatial_groups_roi_collection"].keys()),
        use_corrected=True,
        wavelength_range=(490, 680),
        wavelength_smoothing=10,
        interpolate_wavelengths=[64, 107, 138],
        normalize_method=None,
        legend_loc="outside",
        use_inline_labels=False,
        show_std=False,
    )

    # Restore original ROI collection
    cube.roi_collection = original_rois
else:
    print("⚠️ No spatial groups found in cv_results.")

## STEP 1: Classification (outputs classified_*)

In [ ]:
# Classify segment - outputs classified_bombs, classified_dark, classified_sediment
print("🎯 STEP 1: CLASSIFICATION")
print("=" * 60)

classification_results = cube.classify_segment(
    segment_start=config.UHI_TRACK_RANGE_5[0],
    segment_end=config.UHI_TRACK_RANGE_5[1],
    use_corrected=True,
    quiet=False,
)

print(f"\n✅ Classification complete!")
print(f"   Classes: {classification_results['class_names']}")
print(f"\n📊 Pixel counts:")
for class_name in classification_results["class_names"]:
    count = np.sum(classification_results["classification_map"] == class_name)
    print(f"   {class_name}: {count} pixels")

In [ ]:
# Plot classification (3 classes)
print("📊 Plotting classification map (3 classes)...")

classification_rois = {}
for class_name in classification_results["class_names"]:
    mask = classification_results["classification_map"] == class_name
    rows, cols = np.where(mask)
    pixels = [(col, row + config.UHI_TRACK_RANGE_5[0]) for row, col in zip(rows, cols)]
    if len(pixels) > 0:
        classification_rois[class_name] = pixels

print(f"ROIs created: {list(classification_rois.keys())}")

cube.plot_georef(
    use_corrected=True,
    coordinate_system="NED",
    track_start=config.UHI_TRACK_RANGE_5[0],
    track_end=config.UHI_TRACK_RANGE_5[1],
    figsize=(40, 10),
    roi_collection=classification_rois,
    roi_marker_size=1,
    roi_legend_loc="outside",
    roi_marker_edgewidth=0,
    roi_legend_markersize=50,
)

## STEP 2: Filtering (creates filtered_*)

In [ ]:
# Apply filtering - creates filtered_bombs and filtered_dark
print("\n🧹 STEP 2: FILTERING")
print("=" * 60)

filtering_results = cube.filter_classification(
    classification_map=classification_results["classification_map"],
    class_names=classification_results["class_names"],
    filter_bombs=True,
    filter_dark=True,
    filter_sediment=False,
    min_area_px=10,
    connectivity=8,
    morph_close_radius=0,  # Disabled
    morph_open_radius=0,  # Disabled
    merge_proximity_px=0,
    quiet=False,
)

print(f"\n✅ Filtering complete!")
print(f"   All classes after filtering: {filtering_results['all_classes']}")

In [ ]:
# Plot filtered map (5 classes!)
print("📊 Plotting filtered map (5 classes: classified_* + filtered_*)...")

filtered_rois = {}
for class_name in filtering_results["all_classes"]:
    mask = filtering_results["filtered_map"] == class_name
    rows, cols = np.where(mask)
    pixels = [(col, row + config.UHI_TRACK_RANGE_5[0]) for row, col in zip(rows, cols)]
    if len(pixels) > 0:
        filtered_rois[class_name] = pixels

print(f"ROIs created: {list(filtered_rois.keys())}")
print(f"\n📊 Pixel counts:")
for class_name in sorted(filtered_rois.keys()):
    print(f"   {class_name}: {len(filtered_rois[class_name])} pixels")

cube.plot_georef(
    use_corrected=True,
    coordinate_system="NED",
    track_start=config.UHI_TRACK_RANGE_5[0],
    track_end=config.UHI_TRACK_RANGE_5[1],
    figsize=(40, 10),
    roi_collection=filtered_rois,
    roi_marker_size=1,
    roi_legend_loc="outside",
    roi_marker_edgewidth=0,
    roi_legend_markersize=50,
)

## STEP 3: Merge to create new_sediment

In [ ]:
# Merge filtered classes into new_sediment
print("\n🔀 STEP 3: MERGE FILTERED CLASSES")
print("=" * 60)

merge_results = cube.merge_filtered_to_sediment(
    filtered_map=filtering_results["filtered_map"],
    quiet=False,
)

print(f"\n✅ Merge complete!")
print(f"   Final classes for validation: {merge_results['class_names']}")

In [ ]:
# Plot validation map (3 classes)
print("📊 Plotting validation map (3 classes)...")

validation_map_rois = {}
for class_name in merge_results["class_names"]:
    mask = merge_results["validation_map"] == class_name
    rows, cols = np.where(mask)
    pixels = [(col, row + config.UHI_TRACK_RANGE_5[0]) for row, col in zip(rows, cols)]
    if len(pixels) > 0:
        validation_map_rois[class_name] = pixels

print(f"ROIs created: {list(validation_map_rois.keys())}")
print(f"\n📊 Pixel counts:")
for class_name in sorted(validation_map_rois.keys()):
    print(f"   {class_name}: {len(validation_map_rois[class_name])} pixels")

cube.plot_georef(
    use_corrected=True,
    coordinate_system="NED",
    track_start=config.UHI_TRACK_RANGE_5[0],
    track_end=config.UHI_TRACK_RANGE_5[1],
    figsize=(40, 10),
    roi_collection=validation_map_rois,
    roi_marker_size=1,
    roi_legend_loc="outside",
    roi_marker_edgewidth=0,
    roi_legend_markersize=50,
)

## STEP 4: Validation

In [ ]:
# Validate using the 3-class map
print("\n📊 STEP 4: VALIDATION")
print("=" * 60)

validation_results = cube.validate_classification(
    classification_map=merge_results["validation_map"],
    segment_start=config.UHI_TRACK_RANGE_5[0],
    segment_end=config.UHI_TRACK_RANGE_5[1],
    validation_rois=validation_rois,
    validation_class_mapping={
        "validation_sediment": "new_sediment",
        "validation_dark": "classified_dark",
        "validation_bombs": "classified_bombs",
    },
    quiet=False,
)

# Print summary
if validation_results:
    print(f"\n" + "=" * 60)
    print(f"📊 VALIDATION RESULTS")
    print(f"=" * 60)
    print(f"Overall Accuracy: {validation_results['accuracy']:.3f}")

    print(f"\n📋 Per-Class Results:")
    for class_name in merge_results["class_names"]:
        print(f"\n{class_name}:")
        print(
            f"  Precision: {validation_results['precision_per_class'][class_name]:.3f}"
        )
        print(f"  Recall:    {validation_results['recall_per_class'][class_name]:.3f}")
        print(f"  F1 Score:  {validation_results['f1_per_class'][class_name]:.3f}")
        print(
            f"  Support:   {validation_results['support_per_class'][class_name]} pixels"
        )

In [ ]:
# Plot confusion matrix
if validation_results:
    import matplotlib.pyplot as plt

    cm = validation_results["confusion_matrix"]
    class_names = merge_results["class_names"]

    # Create figure
    fig, ax = plt.subplots(figsize=(10, 8))

    # Plot confusion matrix as image
    im = ax.imshow(cm, cmap="Oranges", aspect="auto")

    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Pixel Count", fontsize=12)

    # Set ticks and labels
    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)

    # Rotate x-axis labels
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    # Add text annotations
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            text = ax.text(
                j,
                i,
                f"{cm[i, j]}",
                ha="center",
                va="center",
                color="black",
                fontsize=12,
            )

    ax.set_xlabel("Predicted Class", fontsize=12)
    ax.set_ylabel("True Class", fontsize=12)
    ax.set_title(
        "Confusion Matrix - Intensity-Only SVM Classification",
        fontsize=14,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

    # Print normalized confusion matrix (percentages)
    cm_normalized = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis] * 100

    print(f"\n📊 Confusion Matrix (Normalized by True Class - Row %):")
    print(f"{'':20s}", end="")
    for pred_class in class_names:
        print(f"{pred_class:>20s}", end="")
    print()
    print("-" * (20 + 20 * len(class_names)))

    for i, true_class in enumerate(class_names):
        print(f"{true_class:20s}", end="")
        for j in range(len(class_names)):
            print(f"{cm_normalized[i, j]:19.1f}%", end="")
        print()

## Summary - Intensity-Only Classification

**Complete workflow executed:**

1. ✅ **Classification** → `classified_bombs`, `classified_dark`, `classified_sediment` (3 classes)
   - **Uses only mean intensity** (no spectral information!)
   - Feature: Single value = mean of wavelengths 490-680 nm
2. ✅ **Filtering** → Creates `filtered_bombs`, `filtered_dark` for small clusters (5 classes total)
3. ✅ **Merge** → `classified_sediment + filtered_* → new_sediment` (3 classes for validation)
4. ✅ **Validation** → Compare against `validation_*` ROIs

**Intensity-Only Mode:**
- 🔬 **Feature**: Mean intensity across wavelength range (490-680 nm)
- 📊 **Dimensionality**: 1 feature (vs ~108 wavelengths in spectral mode)
- 🎯 **Goal**: Test if brightness alone can classify bombs/sediment/dark areas
- ⚡ **Expected**: Likely lower accuracy than spectral classification

**Key improvements:**
- Clear naming: `training_*` (for training) vs `classified_*` (predictions) vs `filtered_*` (removed pixels)
- No empty strings - all pixels have a class
- Can plot 5-class map to see what was filtered
- Can validate against 3-class map
- Morphological operations properly disabled (radius=0)